# L2 latency bench — Kaggle / Colab entry-point

Runs `parapet_runner.latency_bench` against a candidate ONNX classifier under single-thread, no-batching conditions and writes a manifest-bearing JSON result.

**Inputs** (mounted by Kaggle as `/kaggle/input/<dataset>/...`):
* The stratified L2 latency corpus — built locally from v8 curated train+val by `parapet-runner/scripts/build_l2_latency_corpus.py` and uploaded as a private Kaggle dataset.

**Outputs** land under `/kaggle/working/...`.

**Runtime**: CPU. Phase 0.6 in `direction.md` measures CPU latency specifically; GPU runs are exploratory only.

**Two placeholders to substitute** before running:
* `<your-corpus-slug>` — whatever Kaggle dataset slug you uploaded the JSONL under.
* (Optional) the `@<commit_sha>` pin in the install cell — defaults to a pushed `main` commit; bump if you want a different parapet revision.

## 1. Install dependencies

In [ ]:
# parapet-runner[bench] pulls onnxruntime + transformers + sentencepiece.
# Pinned to a specific main commit for reproducibility.
!pip install -q "git+https://github.com/Parapet-Tech/parapet.git@2fa5394c49968aef80053d005a565b2c592cb45d#subdirectory=parapet-runner&egg=parapet-runner[bench]"
!pip install -q "optimum[exporters]>=1.20" "huggingface_hub>=0.25"

## 2. Export mDeBERTa to ONNX

One-shot. Skip on subsequent bench reruns if `/kaggle/working/mdeberta-onnx/model.onnx` already exists.

In [ ]:
!optimum-cli export onnx \
    --model microsoft/mdeberta-v3-base \
    --task feature-extraction \
    --opset 17 \
    /kaggle/working/mdeberta-onnx

## 3. Quantize to int8

direction.md Phase 0.6 measures int8 specifically. ORT's dynamic quantizer needs no calibration dataset.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(
    "/kaggle/working/mdeberta-onnx/model.onnx",
    "/kaggle/working/mdeberta-onnx/model.int8.onnx",
    weight_type=QuantType.QInt8,
)

## 4. Hash the int8 model

`parapet_runner.latency_bench --onnx-sha256` requires the digest so the adapter refuses to load a swapped artifact.

In [ ]:
import hashlib
h = hashlib.sha256()
with open("/kaggle/working/mdeberta-onnx/model.int8.onnx", "rb") as f:
    while chunk := f.read(65536):
        h.update(chunk)
ONNX_SHA = h.hexdigest()
open("/kaggle/working/mdeberta-onnx/model.int8.sha256", "w").write(ONNX_SHA)
print(ONNX_SHA)

## 5. Pin the corpus + model revision

**Corpus**: stratified v8 train/val sample built locally via `python scripts/build_l2_latency_corpus.py`, uploaded to Kaggle as a private dataset. Real curated train/val text — NOT holdout, NOT challenge (see `direction.md` Phase 3 split discipline).

**Model revision**: resolved from the Hub at run time so the manifest pins to a real commit SHA rather than `'main'`.

**TODO before running**: replace `<your-corpus-slug>` below.

In [ ]:
from huggingface_hub import HfApi
CORPUS_PATH = "/kaggle/input/<your-corpus-slug>/l2_latency_v8_train_val_stratified.jsonl"
HF_REVISION = HfApi().model_info("microsoft/mdeberta-v3-base").sha
print(f"resolved mDeBERTa revision: {HF_REVISION}")
print(f"corpus path: {CORPUS_PATH}")

## 6. Run the bench

All bench logic is in `parapet_runner.latency_bench`; this cell just invokes it with the artifacts pinned above.

In [ ]:
import subprocess
subprocess.run([
    "python", "-m", "parapet_runner.latency_bench",
    "--model-path", "/kaggle/working/mdeberta-onnx/model.int8.onnx",
    "--tokenizer-path", "/kaggle/working/mdeberta-onnx",
    "--model-revision", f"microsoft/mdeberta-v3-base@{HF_REVISION}",
    "--onnx-sha256", ONNX_SHA,
    "--quant", "int8",
    "--provider", "CPUExecutionProvider",
    "--corpus", CORPUS_PATH,
    "--output", "/kaggle/working/latency_result.json",
    "--environment", "kaggle",
], check=True)

## 7. Inspect the result

Look for `end_to_end.p50_ms` and `end_to_end.p99_ms`. `direction.md` Phase 0.6 gates: **p50 ≤ 25 ms, p99 ≤ 100 ms**. If the candidate misses both by a wide margin, mDeBERTa is not shippable as the default L2a path — fall back to MiniLM and update `direction.md` accordingly.

In [ ]:
import json
result = json.loads(open("/kaggle/working/latency_result.json").read())
print("end-to-end percentiles (ms):", result["end_to_end"])
print("tokenize percentiles (ms):  ", result["tokenize"])
print("infer percentiles (ms):     ", result["infer"])
print("token length histogram:     ", result["token_length_histogram"])
print("\n--- manifest ---")
print(json.dumps(result["manifest"], indent=2))